# CLAMP BP - Timing (3 runs)

**Environment:** `clamp-analyses`  

Measures SVD, CLAMPbase and CLAMPfull times separately, 3 runs each.

In [1]:
library(bigstatsr)
library(here)
library(CLAMP)

source(here("config.R"))

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



In [2]:
input_dir <- config$GTEx$OUTPUT_DIR
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3
N_CORES <- config$GTEx$N_CORES
base_seed <- config$GTEx$RANDOM_SVD_SEED
seeds <- base_seed + 0:(N_RUNS - 1)

In [3]:
gtex_fbm_filt <- readRDS(file.path(input_dir, "gtex_fbm_filt.rds"))
gtex_genes <- readRDS(file.path(input_dir, "gtex_genes.rds"))

n_genes <- nrow(gtex_fbm_filt)
n_samples <- ncol(gtex_fbm_filt)
SVD_K <- min(n_genes, n_samples) - 1

message("Data: ", n_genes, " genes x ", n_samples, " samples")
message("SVD K = ", SVD_K)

Data: 21613 genes x 17382 samples

SVD K = 17381



In [4]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways



## SVD (3 runs)

In [5]:
SVD_times <- numeric(N_RUNS)
svd_results <- list()
CLAMP_K_values <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("SVD run", i, "of", N_RUNS, "- seed:", seeds[i], "\n")
  
  set.seed(seeds[i])
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  start_time <- Sys.time()
  
  svd_result <- big_randomSVD(gtex_fbm_filt, k = SVD_K, ncores = N_CORES)
  
  end_time <- Sys.time()
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  # Remove NaN values
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  svd_results[[i]] <- svd_result
  
  # Estimate CLAMP K
  CLAMP_K_values[i] <- num.pc(list(d = svd_result$d)) * 2
  
  SVD_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", SVD_times[i], "minutes, CLAMP_K =", CLAMP_K_values[i], "\n\n")
}

SVD run 1 of 3 - seed: 123 
Run 1 time: 510.3514 minutes, CLAMP_K = 412 

SVD run 2 of 3 - seed: 124 
Run 2 time: 507.2705 minutes, CLAMP_K = 412 

SVD run 3 of 3 - seed: 125 
Run 3 time: 485.3173 minutes, CLAMP_K = 412 



In [6]:
SVD_time_minutes <- SVD_times
names(SVD_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(SVD_time_minutes, file.path(output_dir, "SVD_time_minutes.rds"))
cat("SVD times:", SVD_time_minutes, "minutes\n")

SVD times: 510.3514 507.2705 485.3173 minutes


## CLAMPbase (3 runs)

In [7]:
CLAMPbase_times <- numeric(N_RUNS)
base_results <- list()

for (i in 1:N_RUNS) {
  cat("CLAMPbase run", i, "of", N_RUNS, "\n")
  
  set.seed(seeds[i])
  
  start_time <- Sys.time()
  
  gtex_baseRes <- CLAMPbase(
    Y      = gtex_fbm_filt,
    svdres = svd_results[[i]],
    trace  = TRUE,
    clamp_k = CLAMP_K_values[i]
  )
  
  end_time <- Sys.time()
  
  base_results[[i]] <- gtex_baseRes
  CLAMPbase_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", CLAMPbase_times[i], "minutes\n\n")
}

CLAMPbase run 1 of 3 


****

CLAMP k is set to 412

L1 is set to 45.2778145717362

L2 is set to 135.833443715208

Progress 1 / 200 | Bdiff=0.222227, minCor=0.616286

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915387

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958797

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976381

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983586

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987160

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988719

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989041

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989755

Progress 10 / 200 | Bdiff=0.003379, minCor=0.990697

Progress 11 / 200 | Bdiff=0.003044, minCor=0.992363

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994038

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994884

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995539

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995313

Progress 16 / 200 | Bdiff=0.002102, minCor=0.995279

Progress 17 / 200 | Bdiff=0.001984, minCor=0.995572

Progress 18 / 200

Run 1 time: 3.621087 minutes

CLAMPbase run 2 of 3 


****

CLAMP k is set to 412

L1 is set to 45.2778145717362

L2 is set to 135.833443715208

Progress 1 / 200 | Bdiff=0.222227, minCor=0.616286

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915387

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958797

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976381

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983586

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987160

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988719

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989041

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989755

Progress 10 / 200 | Bdiff=0.003379, minCor=0.990697

Progress 11 / 200 | Bdiff=0.003044, minCor=0.992363

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994038

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994884

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995539

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995313

Progress 16 / 200 | Bdiff=0.002102, minCor=0.995279

Progress 17 / 200 | Bdiff=0.001984, minCor=0.995572

Progress 18 / 200

Run 2 time: 3.274057 minutes

CLAMPbase run 3 of 3 


****

CLAMP k is set to 412

L1 is set to 45.2778145717362

L2 is set to 135.833443715208

Progress 1 / 200 | Bdiff=0.222227, minCor=0.616286

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915387

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958797

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976381

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983586

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987160

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988719

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989041

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989755

Progress 10 / 200 | Bdiff=0.003379, minCor=0.990697

Progress 11 / 200 | Bdiff=0.003044, minCor=0.992363

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994038

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994884

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995539

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995313

Progress 16 / 200 | Bdiff=0.002102, minCor=0.995279

Progress 17 / 200 | Bdiff=0.001984, minCor=0.995572

Progress 18 / 200

Run 3 time: 3.356837 minutes



In [8]:
CLAMPbase_time_minutes <- CLAMPbase_times
names(CLAMPbase_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(CLAMPbase_time_minutes, file.path(output_dir, "CLAMPbase_time_minutes.rds"))
cat("CLAMPbase times:", CLAMPbase_time_minutes, "minutes\n")

CLAMPbase times: 3.621087 3.274057 3.356837 minutes


## CLAMPfull with BP prior (3 runs)

In [9]:
CLAMPfull_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("CLAMPfull run", i, "of", N_RUNS, "\n")
  
  set.seed(seeds[i])
  
  start_time <- Sys.time()
  
  gtex_fullRes <- CLAMPfull(
    Y                 = gtex_fbm_filt,
    priorMat          = as.matrix(gtex_matched),
    svdres            = svd_results[[i]],
    clamp.base.result = base_results[[i]],
    clamp_k           = CLAMP_K_values[i],
    doCrossval        = TRUE,
    trace             = TRUE,
    use_cpp           = TRUE
  )
  
  end_time <- Sys.time()
  CLAMPfull_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", CLAMPfull_times[i], "minutes\n\n")
}

CLAMPfull run 1 of 3 


** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2778145717362; L2=135.833443715208

Progress 1 / 30 | Bdiff=0.000454

Progress 2 / 30 | Bdiff=0.037538

Progress 3 / 30 | Bdiff=0.015136

Estimated total runtime: ~7.5 min

Progress 4 / 30 | Bdiff=0.012534

Progress 5 / 30 | Bdiff=0.008689

Progress 6 / 30 | Bdiff=0.006457

Progress 7 / 30 | Bdiff=0.005420

Progress 8 / 30 | Bdiff=0.005177

Progress 9 / 30 | Bdiff=0.005168

Progress 10 / 30 | Bdiff=0.004985

Progress 11 / 30 | Bdiff=0.004880

Progress 12 / 30 | Bdiff=0.004480

Progress 13 / 30 | Bdiff=0.004017

Progress 14 / 30 | Bdiff=0.004073

Progress 15 / 30 | Bdiff=0.003898

Progress 16 / 30 | Bdiff=0.003608

Progress 17 / 30 | Bdiff=0.003637

Progress 18 / 30 | Bdiff=0.003338

Progress 19 / 30 | Bdiff=0.003323

Progress 20 / 30 | Bdiff=0.003197

Progress 21 / 30 | Bdiff=0.003059

Progress 22 / 30 | Bdiff=0.003168

Progress 23 / 30 | Bdiff=0.003308

Progress 24 / 30 | Bdiff=0.003360

Progress 25 / 30 | 

Run 1 time: 13.54375 minutes

CLAMPfull run 2 of 3 


** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2778145717362; L2=135.833443715208

Progress 1 / 30 | Bdiff=0.000454

Progress 2 / 30 | Bdiff=0.037533

Progress 3 / 30 | Bdiff=0.015008

Estimated total runtime: ~7.5 min

Progress 4 / 30 | Bdiff=0.012581

Progress 5 / 30 | Bdiff=0.008713

Progress 6 / 30 | Bdiff=0.006532

Progress 7 / 30 | Bdiff=0.005407

Progress 8 / 30 | Bdiff=0.005148

Progress 9 / 30 | Bdiff=0.005050

Progress 10 / 30 | Bdiff=0.004873

Progress 11 / 30 | Bdiff=0.004869

Progress 12 / 30 | Bdiff=0.004429

Progress 13 / 30 | Bdiff=0.003981

Progress 14 / 30 | Bdiff=0.003945

Progress 15 / 30 | Bdiff=0.003787

Progress 16 / 30 | Bdiff=0.003569

Progress 17 / 30 | Bdiff=0.003599

Progress 18 / 30 | Bdiff=0.003273

Progress 19 / 30 | Bdiff=0.003279

Progress 20 / 30 | Bdiff=0.003321

Progress 21 / 30 | Bdiff=0.003185

Progress 22 / 30 | Bdiff=0.003245

Progress 23 / 30 | Bdiff=0.003499

Progress 24 / 30 | Bdiff=0.003596

Progress 25 / 30 | 

Run 2 time: 13.38874 minutes

CLAMPfull run 3 of 3 


** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2778145717362; L2=135.833443715208

Progress 1 / 30 | Bdiff=0.000454

Progress 2 / 30 | Bdiff=0.037470

Progress 3 / 30 | Bdiff=0.015068

Estimated total runtime: ~7.5 min

Progress 4 / 30 | Bdiff=0.012629

Progress 5 / 30 | Bdiff=0.008896

Progress 6 / 30 | Bdiff=0.006564

Progress 7 / 30 | Bdiff=0.005411

Progress 8 / 30 | Bdiff=0.004997

Progress 9 / 30 | Bdiff=0.004902

Progress 10 / 30 | Bdiff=0.004827

Progress 11 / 30 | Bdiff=0.004943

Progress 12 / 30 | Bdiff=0.004559

Progress 13 / 30 | Bdiff=0.004083

Progress 14 / 30 | Bdiff=0.004153

Progress 15 / 30 | Bdiff=0.003811

Progress 16 / 30 | Bdiff=0.003640

Progress 17 / 30 | Bdiff=0.003570

Progress 18 / 30 | Bdiff=0.003202

Progress 19 / 30 | Bdiff=0.003163

Progress 20 / 30 | Bdiff=0.003181

Progress 21 / 30 | Bdiff=0.003089

Progress 22 / 30 | Bdiff=0.003331

Progress 23 / 30 | Bdiff=0.003534

Progress 24 / 30 | Bdiff=0.003604

Progress 25 / 30 | 

Run 3 time: 13.15536 minutes



In [10]:
CLAMPfull_BP_time_minutes <- CLAMPfull_times
names(CLAMPfull_BP_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(CLAMPfull_BP_time_minutes, file.path(output_dir, "CLAMPfull_BP_time_minutes.rds"))
cat("CLAMPfull (BP) times:", CLAMPfull_BP_time_minutes, "minutes\n")

CLAMPfull (BP) times: 13.54375 13.38874 13.15536 minutes


## Summary

In [11]:
cat("\n=== Timing Summary (minutes) ===\n")
cat("SVD:       ", paste(round(SVD_time_minutes, 2), collapse = ", "), "\n")
cat("CLAMPbase: ", paste(round(CLAMPbase_time_minutes, 2), collapse = ", "), "\n")
cat("CLAMPfull: ", paste(round(CLAMPfull_BP_time_minutes, 2), collapse = ", "), "\n")
cat("\nTotal per run:", paste(round(SVD_time_minutes + CLAMPbase_time_minutes + CLAMPfull_BP_time_minutes, 2), collapse = ", "), "\n")


=== Timing Summary (minutes) ===
SVD:        510.35, 507.27, 485.32 
CLAMPbase:  3.62, 3.27, 3.36 
CLAMPfull:  13.54, 13.39, 13.16 

Total per run: 527.52, 523.93, 501.83 
